# Quant Portfolio Lab — Exploratory Analysis (v0.1)

A guided tour of the platform on **synthetic data** (runs fully offline):

1. Generate a synthetic KR-style market
2. Inspect prices and the benchmark
3. Build point-in-time PER/PBR factor snapshots
4. Rank the universe (lowest PBR)
5. Run the low-PBR backtest and view the result page
6. Inspect the transaction-cost model

> Research only · not investment advice.

In [ ]:
import sys
from pathlib import Path

# Make the package importable when running from the repo without installing.
SRC = Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

from quant_portfolio_lab.data.synthetic import make_synthetic_market
from quant_portfolio_lab.factors.valuation import build_factor_snapshots, rank_by_factor
from quant_portfolio_lab.backtest.cost_model import CostModel
from quant_portfolio_lab.backtest.engine import BacktestConfig, BacktestEngine
from quant_portfolio_lab.visualization.charts import (
    cumulative_return_chart, drawdown_chart, annual_return_table,
)

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

## 1. Generate a synthetic market

In [ ]:
market = make_synthetic_market(n_assets=12, start="2018-01-01", end="2024-12-31")
print(market.assets.shape, market.prices.shape, market.fundamentals.shape)
market.assets.head()

In [ ]:
price_panel = market.prices.pivot(index="date", columns="asset_id", values="close")
price_panel.index = pd.DatetimeIndex(price_panel.index)
benchmark = market.benchmark.set_index("date")["close"]
price_panel.tail()

## 2. Inspect a few price series

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
for asset_id in price_panel.columns[:4]:
    s = price_panel[asset_id]
    fig.add_trace(go.Scatter(x=s.index, y=s.values, name=str(asset_id)))
fig.update_layout(title="Sample synthetic prices", template="plotly_white")
fig.show()

## 3. Point-in-time factor snapshots

Only fundamentals with `report_date <= as_of_date` are used, and EPS/BPS ≤ 0 or
missing values are excluded: this is what prevents look-ahead bias.

In [ ]:
as_of = pd.Timestamp("2022-01-03")
snapshots = build_factor_snapshots(price_panel, market.fundamentals, as_of)
snapshots

## 4. Rank by lowest PBR

In [ ]:
ranked = rank_by_factor(snapshots, factor="pbr", ascending=True)
ranked[["rank", "asset_id", "price", "bps", "pbr", "valid_for_backtest"]].head(10)

## 5. Run the low-PBR backtest

In [ ]:
engine = BacktestEngine(
    price_panel,
    CostModel(fee_rate=0.001, tax_rate=0.0020, slippage_rate=0.002),
    fundamentals=market.fundamentals,
    benchmark=benchmark,
    config=BacktestConfig(rebalance_mode="1Y", top_n=5, factor="pbr", benchmark_id="KOSPI"),
)
result = engine.run()
result.metrics

In [ ]:
cumulative_return_chart(result.equity_curve, result.benchmark).show()

In [ ]:
drawdown_chart(result.equity_curve).show()

In [ ]:
annual_return_table(result.equity_curve, result.benchmark).show()

In [ ]:
print(f"trades: {len(result.trades)}, avg turnover: {result.turnover:.2%}")
result.trades.head()

## 6. Transaction-cost model sanity check

In [ ]:
cm = CostModel(fee_rate=0.001, tax_rate=0.0020, slippage_rate=0.002)
buy = cm.price_trade("BUY", quantity=10, close_price=10_000)
sell = cm.price_trade("SELL", quantity=10, close_price=10_000)
print("BUY :", buy)
print("SELL:", sell)
print("buy exec  = 10000 * (1 + 0.002) =", buy.execution_price)
print("sell exec = 10000 * (1 - 0.002) =", sell.execution_price)
print("sell tax (sell-side only)       =", sell.transaction_tax)